# Nested Stratified Group Cross-Validation — IQ-OTH/NCCD

**Purpose:** Evaluate model selection among the original **17** spatial patch configurations using outer 5-fold and inner 3-fold similarity-grouped CV.

**Inputs via Kaggle Add Input:** `STAGE2_RESULTS_SMALL.zip`, `STAGE2_FEATURE_CACHE.zip`, and the **same original IQ-OTH/NCCD JPG dataset**. Enable a T4 GPU for feature extraction.

**Important:** This is **not patient-independent validation** because verified patient IDs are unavailable. It also should not be described as a pristine preregistered test because the candidate set was previously explored.


In [ ]:
import os
from pathlib import Path
print('GPU available:', __import__('torch').cuda.is_available())
for name in ('STAGE2_RESULTS_SMALL.zip','STAGE2_FEATURE_CACHE.zip'):
    paths=list(Path('/kaggle/input').rglob(name))+list(Path('/kaggle/working').rglob(name))
    print(name, 'FOUND' if paths else 'MISSING', paths[:2])
print('JPG count:',sum(1 for _ in Path('/kaggle/input').rglob('*.jpg')))

## Run nested evaluation

Run the next cell. It extracts 16-patch features once, then fits PCA, patch-selection procedures, scaler and SVM *inside* the appropriate training folds. Progress is saved after each outer fold so it can be resumed in the same session. Download `NESTED_GROUP_CV_RESULTS.zip` after **DONE**.


In [ ]:
import os, json, zipfile, hashlib, time, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, recall_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import mutual_info_score

DATA_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
OUT = WORK_ROOT / 'NESTED_GROUP_CV_RESULTS'
OUT.mkdir(parents=True, exist_ok=True)
RANDOM_STATE=42
CLASSES=['Benign','Malignant','Normal']

CANDIDATES={
  16:[('mean',0),('all',16),('relevance',4),('random',8),('relevance',8),('MID',8),('MIQ',8),('relevance',12)],
  64:[('mean',0),('all',64),('relevance',8),('relevance',16),('random',20),('relevance',20),('MID',20),('MIQ',20),('relevance',32)]
}
assert sum(len(v) for v in CANDIDATES.values())==17

def locate(name):
    # Searches files actually mounted by Kaggle in current notebook.
    candidates=[]
    for top in (WORK_ROOT, DATA_ROOT):
        direct=top/name
        if direct.is_file(): candidates.append(direct)
        candidates.extend(p for p in top.rglob(name) if p.is_file())
    seen=set(); found=[]
    for p in candidates:
        if str(p) not in seen: found.append(p);seen.add(str(p))
    return found[0] if found else None

def extract_required_zip(zip_name, required_names):
    missing=[n for n in required_names if locate(n) is None]
    if not missing: return
    p=locate(zip_name)
    if p is None:
        raise FileNotFoundError(f'Missing {missing}; attach {zip_name} through Kaggle Add Input (or keep the files in /kaggle/working).')
    dest=WORK_ROOT / ('stage2_cache' if 'FEATURE' in zip_name else 'stage2_small')
    dest.mkdir(exist_ok=True)
    with zipfile.ZipFile(p) as z:
        index={Path(n).name:n for n in z.namelist() if not n.endswith('/')}
        for n in missing:
            if n not in index: raise FileNotFoundError(f'{n} absent inside {zip_name}.')
            target=dest/n
            if not target.exists():
                with z.open(index[n]) as src, target.open('wb') as out:
                    import shutil;shutil.copyfileobj(src,out)
    print('Extracted:',', '.join(missing), flush=True)

extract_required_zip('STAGE2_RESULTS_SMALL.zip', ['stage2_manifest.csv','labels.npy','groups.npy','folds.npy'])
extract_required_zip('STAGE2_FEATURE_CACHE.zip', ['features_64_224.npy','whole_features_224.npy'])

manifest = pd.read_csv(locate('stage2_manifest.csv'))
y=np.load(locate('labels.npy'))
groups=np.load(locate('groups.npy'))
folds=np.load(locate('folds.npy'))
X64=np.load(locate('features_64_224.npy'), mmap_mode='r')
Xwhole=np.load(locate('whole_features_224.npy'), mmap_mode='r')
N=len(y)
assert N==1054 and len(manifest)==N and X64.shape==(N,64,1024) and Xwhole.shape==(N,1024)
assert np.array_equal(y,manifest.label_code.to_numpy())
assert np.array_equal(groups,manifest.similarity_group.to_numpy())
assert np.array_equal(folds,manifest.fold.to_numpy())
assert set(np.unique(folds))=={1,2,3,4,5}
for o in range(1,6):
    a=folds!=o; b=folds==o
    assert not set(groups[a])&set(groups[b]),'Similarity-group overlap in input folds'
print('AUDIT OK',N,'images;',len(np.unique(groups)),'groups;',dict(Counter(y)),'class labels',flush=True)

# Ensure that the source images needed for the 16-patch cache match the manifest in the exact same order.
image_paths=[Path(p) for p in manifest.path]
missing=[str(p) for p in image_paths if not p.is_file()]
if missing:
    print('One example missing path:',missing[0])
    raise FileNotFoundError('Original IQ-OTH/NCCD JPG data must also be added to Kaggle Input at the paths in stage2_manifest.csv.')
for p,h in zip(image_paths, manifest.sha256.astype(str)):
    if hashlib.sha256(p.read_bytes()).hexdigest()!=h:
        raise AssertionError(f'Image content changed: {p}')
print('SHA-256 integrity verified for all 1054 images.',flush=True)

# 16-patch features are not in STAGE2_FEATURE_CACHE; generate them once, using exactly the
# grayscale/PIL preprocessing used for the saved 64-patch feature cache.
cache16=WORK_ROOT/'features_16_224_nested.npy'
if cache16.exists():
    X16=np.load(cache16,mmap_mode='r')
    assert X16.shape==(N,16,1024)
    print('Reusing cached 16-patch features.',flush=True)
else:
    import torch
    import torch.nn as nn
    from PIL import Image
    from torchvision import transforms as T
    from torchvision.models import googlenet, GoogLeNet_Weights
    weights=GoogLeNet_Weights.IMAGENET1K_V1
    try:
        model=googlenet(weights=weights)
    except Exception as e:
        local=locate('googlenet-1378be20.pth')
        if local is None:
            raise RuntimeError('GoogLeNet pretrained weights are unavailable. Turn on Kaggle Internet OR add googlenet-1378be20.pth to Kaggle Input.') from e
        model=googlenet(weights=None,aux_logits=True,init_weights=False)
        model.load_state_dict(torch.load(local,map_location='cpu',weights_only=True),strict=True)
        model.aux1=None;model.aux2=None;model.aux_logits=False
    model.fc=nn.Identity()
    model.eval().to('cuda' if torch.cuda.is_available() else 'cpu')
    device=next(model.parameters()).device
    trans=T.Compose([T.Resize((224,224)),T.Grayscale(num_output_channels=3),T.ToTensor(),
        T.Normalize([.485,.456,.406],[.229,.224,.225])])
    X16write=np.lib.format.open_memmap(cache16,mode='w+',dtype='float32',shape=(N,16,1024))
    t0=time.time()
    with torch.inference_mode():
        for i,p in enumerate(image_paths):
            im=Image.open(p).convert('L').resize((224,224))
            patches=[trans(im.crop((56*c,56*r,56*(c+1),56*(r+1)))) for r in range(4) for c in range(4)]
            ten=torch.stack(patches).to(device)
            result=model(ten)
            if not torch.is_tensor(result): result=result.logits
            X16write[i]=result.cpu().numpy().astype('float32')
            if (i+1)%100==0 or i==N-1: print('16-patch extraction',i+1,'/',N,flush=True)
    X16write.flush()
    X16=np.load(cache16,mmap_mode='r')
    del model, X16write
    if torch.cuda.is_available():torch.cuda.empty_cache()
    print('16-patch features saved:',cache16,'minutes',round((time.time()-t0)/60,2),flush=True)

assert X16.shape==(N,16,1024)
FEATURES={16:X16,64:X64}

def pca_patch_train_test(X,train_idx,test_idx):
    G=X.shape[1]
    xtr=np.asarray(X[train_idx],dtype=np.float32)
    xte=np.asarray(X[test_idx],dtype=np.float32)
    pca=PCA(n_components=64,svd_solver='randomized',random_state=42)
    A=pca.fit_transform(xtr.reshape(-1,1024)).reshape(len(train_idx),G,64)
    B=pca.transform(xte.reshape(-1,1024)).reshape(len(test_idx),G,64)
    return A,B

def compute_ranking(A,ytrain):
    # PCA projection per position; quantile bins fit on training samples only.
    n,G,d=A.shape
    binned=np.empty((n,G),dtype='uint8')
    for j in range(G):
        a=np.asarray(A[:,j,:],dtype=np.float64)
        cen=a-a.mean(axis=0)
        eig,vec=np.linalg.eigh(cen.T@cen)
        proj=cen@vec[:,-1]
        q=np.quantile(proj,np.arange(1,8)/8)
        binned[:,j]=np.searchsorted(q,proj,side='right')
    relevance=np.array([mutual_info_score(binned[:,i],ytrain) for i in range(G)])
    redund=np.zeros((G,G),dtype=float)
    for i in range(G):
        for j in range(i+1,G):
            redund[i,j]=redund[j,i]=mutual_info_score(binned[:,i],binned[:,j])
    return relevance,redund

def choose_pos(R,D,method,k):
    rem=list(range(len(R)));chosen=[]
    while len(chosen)<k:
        def score(j):
            if not chosen or method=='relevance': return R[j]
            avg=D[j,chosen].mean()
            if method=='MID':return R[j]-avg
            if method=='MIQ':return R[j]/(avg+0.001)
            raise ValueError(method)
        best=min(rem,key=lambda j:(-score(j),j))
        chosen.append(best);rem.remove(best)
    return np.array(sorted(chosen),dtype=int)

def train_predict(A,B,ytr,kind,k,R,D,seed,foldtag):
    G=A.shape[1]
    if kind=='mean': a=A.mean(axis=1);b=B.mean(axis=1)
    elif kind=='all':a=A.reshape(len(A),-1);b=B.reshape(len(B),-1)
    else:
        if kind=='random':sel=np.sort(np.random.default_rng(seed+G*100+foldtag).choice(G,size=k,replace=False))
        else:sel=choose_pos(R,D,kind,k)
        a=A[:,sel,:].reshape(len(A),-1)
        b=B[:,sel,:].reshape(len(B),-1)
    sc=StandardScaler();a=sc.fit_transform(a);b=sc.transform(b)
    clf=SVC(C=1,gamma='scale',kernel='rbf',class_weight='balanced')
    clf.fit(a,ytr)
    return clf.predict(b)

def whole_predict(train,test):
    pca=PCA(n_components=64,random_state=42)
    a=pca.fit_transform(Xwhole[train]);b=pca.transform(Xwhole[test])
    sc=StandardScaler();a=sc.fit_transform(a);b=sc.transform(b)
    clf=SVC(C=1,gamma='scale',kernel='rbf',class_weight='balanced')
    clf.fit(a,y[train]);return clf.predict(b)

# Outer folds: exactly those saved from Stage 2. Each inner split is group-separated,
# and PCA, selector, scaler, SVM are refitted on INNER-training images only.
outer_pred=np.full(N,-1,dtype=int)
full64_pred=np.full(N,-1,dtype=int)
whole_pred=np.full(N,-1,dtype=int)
log=[]
start=time.time()
for outer in range(1,6):
    progress=OUT/f'outer_fold_{outer}.npz'
    if progress.exists():
        with np.load(progress,allow_pickle=False) as z:
            test=z['test'];p=z['selected'];q=z['full64'];w=z['whole'];best=str(z['chosen'])
        outer_pred[test]=p;full64_pred[test]=q;whole_pred[test]=w
        log.append({'outer_fold':outer,'selected_candidate':best,'test_n':int(len(test))})
        print('Resumed outer fold',outer, 'selected',best,flush=True)
        continue
    test=np.where(folds==outer)[0]
    develop=np.where(folds!=outer)[0]
    inner=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=2026+outer)
    inner_splits=[(develop[tr],develop[val]) for tr,val in inner.split(np.zeros(len(develop)),y[develop],groups[develop])]
    for tr,val in inner_splits: assert not set(groups[tr])&set(groups[val])
    oof={f'{G}:{method}:{k}':np.full(len(develop),-1,dtype=int)
         for G,settings in CANDIDATES.items() for method,k in settings}
    local_lookup={idx:i for i,idx in enumerate(develop)}
    for inner_id,(tr,val) in enumerate(inner_splits,1):
        position=np.array([local_lookup[int(i)] for i in val],dtype=int)
        for G in (16,64):
            A,B=pca_patch_train_test(FEATURES[G],tr,val)
            R,D=compute_ranking(A,y[tr])
            for method,k in CANDIDATES[G]:
                key=f'{G}:{method}:{k}'
                oof[key][position]=train_predict(A,B,y[tr],method,k,R,D,42,outer*100+inner_id)
        print(f'Outer {outer}/5, inner {inner_id}/3 complete',flush=True)
    scores={key:float(f1_score(y[develop],p,average='macro',zero_division=0)) for key,p in oof.items()}
    assert all(np.all(p>=0) for p in oof.values())
    # Deterministic tie-break. The candidate is selected WITHOUT seeing outer-test labels.
    best=sorted(scores,key=lambda key:(-scores[key],key))[0]
    winner_G,winner_method,winner_k=best.split(':');winner_G=int(winner_G);winner_k=int(winner_k)
    outer_cache={}
    for G in (16,64) if winner_G==16 else (64,):
        A,B=pca_patch_train_test(FEATURES[G],develop,test)
        outer_cache[G]=(A,B)
    A,B=outer_cache[winner_G]
    R,D=compute_ranking(A,y[develop])
    p=train_predict(A,B,y[develop],winner_method,winner_k,R,D,42,9000+outer)
    A64,B64=outer_cache.get(64,(None,None))
    if A64 is None:A64,B64=pca_patch_train_test(FEATURES[64],develop,test)
    q=train_predict(A64,B64,y[develop],'all',64,None,None,42,9000+outer)
    w=whole_predict(develop,test)
    outer_pred[test]=p;full64_pred[test]=q;whole_pred[test]=w
    pd.DataFrame([{'outer_fold':outer,'candidate':key,'inner_macro_f1':round(score,6)} for key,score in scores.items()]).to_csv(OUT/f'inner_scores_fold_{outer}.csv',index=False)
    np.savez_compressed(progress,test=test,selected=p,full64=q,whole=w,chosen=np.asarray(best))
    log.append({'outer_fold':outer,'selected_candidate':best,'selected_inner_macro_f1':scores[best],
      'test_n':len(test),'selected_outer_macro_f1':f1_score(y[test],p,average='macro',zero_division=0)})
    (OUT/'progress.json').write_text(json.dumps(log,indent=2))
    print(f'OUTER {outer}/5 complete. Selected {best}. Outer macro-F1 {f1_score(y[test],p,average="macro"):.4f}',flush=True)
assert np.all(outer_pred>=0) and np.all(full64_pred>=0) and np.all(whole_pred>=0)
models={'Nested selection among original 17':outer_pred,
        'Fixed all 64 patches':full64_pred,
        'Whole-image baseline':whole_pred}
res=[]
for name,pred in models.items():
    row={'model':name,'accuracy_pct':100*accuracy_score(y,pred),
         'macro_f1_pct':100*f1_score(y,pred,average='macro'),
         'balanced_accuracy_pct':100*balanced_accuracy_score(y,pred),
         'benign_recall_pct':100*recall_score(y,pred,labels=[0],average=None)[0]}
    res.append(row)
    print('RESULT',row,flush=True)
pd.DataFrame(res).to_csv(OUT/'nested_summary.csv',index=False)
pd.DataFrame({'path':manifest.path,'true':y,'group':groups,'outer_fold':folds,
              'nested_pred':outer_pred,'fixed64_pred':full64_pred,'whole_pred':whole_pred}).to_csv(OUT/'nested_predictions.csv',index=False)
pd.DataFrame(log).to_csv(OUT/'selected_models_by_fold.csv',index=False)
(OUT/'nested_protocol.json').write_text(json.dumps({
 'dataset_n':N,'class_counts':dict(Counter(map(str,y))),
 'similarity_groups':int(len(np.unique(groups))),'outer_folds':5,'inner_folds':3,
 'number_of_candidate_models':17,'selection_metric':'pooled inner out-of-fold macro-F1',
 'outer_metric':'pooled outer out-of-fold classification',
 'candidate_models':{str(k):[f'{a}:{b}' for a,b in v] for k,v in CANDIDATES.items()},
 'warning':'NO verified patient IDs; results remain similarity-grouped image-level evaluation. Internal model-selection protocol was designed after seeing prior Stage 2 results; this does NOT constitute a pristine independently pre-registered validation.',
 },indent=2))
with zipfile.ZipFile(WORK_ROOT/'NESTED_GROUP_CV_RESULTS.zip','w',zipfile.ZIP_DEFLATED) as z:
    for path in OUT.glob('*'):
        if path.is_file():z.write(path,path.name)
print('DONE. Download: /kaggle/working/NESTED_GROUP_CV_RESULTS.zip',flush=True)
print('Minutes elapsed:',round((time.time()-start)/60,2),flush=True)
